# Summary

Test calling a deployed agent

In [1]:
import os, sys
import json

from vertexai import agent_engines

utils_path = "../../../../ccc-policy_assistant/interface/utils"
sys.path.insert(0, utils_path)
from authentication import ApiAuthentication

# Set environment variables
dotenv_path = "../../../../ccc-policy_assistant/data/environment"
api_configs = ApiAuthentication(dotenv_path=dotenv_path)
api_configs.set_environ_variables()



## Retrieve a deployed agent

In [2]:

# Retreive an ADK agent
agent_engine = agent_engines.get(os.getenv("AGENT_ENGINE_ID"))

# dir(agent_engine)
# Create a session
session = agent_engine.create_session(user_id="u_123")


In [4]:
dir(agent_engine)

# Create details on ways to interact with the agent
ops = agent_engine.operation_schemas()

print("These are the actions that operations that can the agent can perform (see dictionary entry for full details on each op):")
for op in ops:
    print(op["name"])


# dir(agent_engine)
# agent_engine.credentials
user_id = "U_123"
agent_engine.list_sessions(user_id=user_id)

These are the actions that operations that can the agent can perform (see dictionary entry for full details on each op):
get_session
list_sessions
create_session
delete_session
stream_query
streaming_agent_run_with_events


{'sessions': [{'events': [],
   'user_id': 'U_123',
   'state': {},
   'id': '7548363378717097984',
   'app_name': '907343383519821824',
   'last_update_time': 1745710676.400482},
  {'events': [],
   'last_update_time': 1745710365.506737,
   'state': {},
   'id': '441683166726455296',
   'app_name': '907343383519821824',
   'user_id': 'U_123'},
  {'events': [],
   'last_update_time': 1745709972.123901,
   'state': {},
   'id': '8800364075126095872',
   'app_name': '907343383519821824',
   'user_id': 'U_123'},
  {'events': [],
   'user_id': 'U_123',
   'state': {},
   'id': '3729310894706917376',
   'app_name': '907343383519821824',
   'last_update_time': 1745709539.386339},
  {'events': [],
   'last_update_time': 1745706279.067282,
   'state': {},
   'id': '4882232399313764352',
   'app_name': '907343383519821824',
   'user_id': 'U_123'}]}

## Test the agent with one query

In [5]:
test_result = agent_engine.stream_query(message="What are LEAP exams?", user_id="U_123")

for event in test_result:
    print(event)


{'content': {'parts': [{'text': 'LEAP exams are alternate examinations and an appointment process designed for recruiting and hiring individuals with disabilities into State service. To become certified for LEAP, you should contact the Department of Rehabilitation.\n'}], 'role': 'model'}, 'grounding_metadata': {'grounding_chunks': [{'retrieved_context': {'text': "We share responsibility for creating an equitable, diverse and inclusive community and we see these values as connected to our mission and critical to ensure the well-being of our staff and the students we serve. For more information on the Chancellor's Office Diversity, Equity, Inclusion and Accessibility (DEIA) efforts, read our DEIA Strategic Plan (PDF). CalCareers hosts our job postings and assessment examinations for Board of Governors, California Community Colleges. We administer Limited Examination Appointment Program (LEAP) Exams for all our department specific classifications. LEAP exams are an alternate examination a

## Test the agent with multiple queries in a session

In [6]:
def pretty_print_event(event):
    """Pretty prints an event with truncation for long content."""
    if "content" not in event:
        print(f"[{event.get('author', 'unknown')}]: {event}")
        return

    author = event.get("author", "unknown")
    parts = event["content"].get("parts", [])

    for part in parts:
        if "text" in part:
            text = part["text"]
            # Truncate long text to 200 characters
            if len(text) > 200:
                text = text[:197] + "..."
            print(f"[{author}]: {text}")
        elif "functionCall" in part:
            func_call = part["functionCall"]
            print(f"[{author}]: Function call: {func_call.get('name', 'unknown')}")
            # Truncate args if too long
            args = json.dumps(func_call.get("args", {}))
            if len(args) > 100:
                args = args[:97] + "..."
            print(f"  Args: {args}")
        elif "functionResponse" in part:
            func_response = part["functionResponse"]
            print(f"[{author}]: Function response: {func_response.get('name', 'unknown')}")
            # Truncate response if too long
            response = json.dumps(func_response.get("response", {}))
            if len(response) > 100:
                response = response[:97] + "..."
            print(f"  Response: {response}")

agent_engine = agent_engines.get(os.getenv("AGENT_ENGINE_ID"))
print(f"Agent ID: Id='{agent_engine.resource_name}'")

# Create a session
user_id = "u_123"
session = agent_engine.create_session(user_id=user_id)
print(f"Session created: User='{user_id}', Session='{session['id']}'")

# Create some queries
queries = [
    "Hi, how are you?",
    "What are LEAP exams?",
    "What is the California Community Colleges Chancellor’s Office doing to ensure accessibility of its websites"
]

# Look for events for each query
for query in queries:
    print(f"\n[user]: {query}")
    for event in agent_engine.stream_query(
        user_id=user_id,
        session_id=session["id"],
        message=query,
    ):
        print("We got here by finding an event")
        pretty_print_event(event)


Agent ID: Id='projects/1062597788108/locations/us-central1/reasoningEngines/907343383519821824'
Session created: User='u_123', Session='8073032735305760768'

[user]: Hi, how are you?
We got here by finding an event
[ask_rag_agent]: I am doing well, thank you for asking! How are you today?


[user]: What are LEAP exams?
We got here by finding an event
[ask_rag_agent]: LEAP exams are an alternate examination and appointment process for the recruitment and hiring of individuals with disabilities into State service. Please contact the Department of Rehabilitation f...

[user]: What is the California Community Colleges Chancellor’s Office doing to ensure accessibility of its websites
We got here by finding an event
[ask_rag_agent]: The California Community Colleges Chancellor’s Office is committed to ensuring digital accessibility for people with disabilities by continually improving the user experience and applying relevant ...


## Look at the sessions

In [7]:
agent_engine.list_sessions(user_id=user_id)

{'sessions': [{'events': [],
   'last_update_time': 1745866004.030116,
   'state': {},
   'id': '8073032735305760768',
   'app_name': '907343383519821824',
   'user_id': 'u_123'},
  {'events': [],
   'user_id': 'u_123',
   'state': {},
   'id': '7633931771637137408',
   'app_name': '907343383519821824',
   'last_update_time': 1745865884.255764},
  {'events': [],
   'user_id': 'u_123',
   'state': {},
   'id': '8115816931765780480',
   'app_name': '907343383519821824',
   'last_update_time': 1745710688.930603},
  {'events': [],
   'user_id': 'u_123',
   'state': {},
   'id': '2981713356563415040',
   'app_name': '907343383519821824',
   'last_update_time': 1745710667.119153},
  {'events': [],
   'last_update_time': 1745710081.368453,
   'state': {},
   'id': '8340996913134305280',
   'app_name': '907343383519821824',
   'user_id': 'u_123'},
  {'events': [],
   'last_update_time': 1745709980.298746,
   'state': {},
   'id': '5954089110627942400',
   'app_name': '907343383519821824',
   '